<a href="https://colab.research.google.com/github/Matheusbcy/-Data-Science-IA-/blob/main/Gerador%20de%20conte%C3%BAdo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto para Departamento de Marketing

## Instalação das bibliotecas

In [ ]:
!pip install -q langchain langchain-community langchain-groq ipywidgets

## Importação

In [2]:
from google.colab import widgets
import ipywidgets as widgets

from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import os
import getpass

# Criação dos campos - interface

## Ajustando propriedades do campo

In [ ]:
topic = widgets.Text(
    description = "Tema:",
    placeholder = "Ex: Saúde mental, alimentação saudável, prevenção, etc.",
    layout = widgets.Layout(width = "500px")
)

display(topic)

## Outros formatos de campos

In [ ]:
w_dropdown = "250px"

platform = widgets.Dropdown(
    options = ["Instagram", "Facebook", "Linkedln", "Blog", "E-mail"],
    description = "Plataforma: ",
    layout = widgets.Layout(width = w_dropdown)
)

tone = widgets.Dropdown(
    options = ["Normal", "Informativo", "Inspirador", "Urgente", "Informal"],
    description = "Tom: ",
    layout = widgets.Layout(width = w_dropdown)
)

length = widgets.Dropdown(
    options = ["Curto", "Médio", "Longo"],
    description = "Tamanho: ",
    layout = widgets.Layout(width = w_dropdown)
)

audience = widgets.Dropdown(
    options = ["Geral", "Jovens adultos", "Famílias", "Idosos", "Adolescentes"],
    description = "Público-alvo: ",
    layout = widgets.Layout(width = w_dropdown)
)

display(platform, tone, length, audience)

In [49]:
cta = widgets.Checkbox(
    value = False,
    description = "Incluir CTA"
)

emoji = widgets.Checkbox(
    value = False,
    description = "Incluir Emojis"
)

hashtags = widgets.Checkbox(
    value = False,
    description = "Retornar Hashtags"
)

In [6]:
keywords = widgets.Textarea(
    description = "Palavras-chave (SEO)",
    placeholder = "Ex: bem-estar, medicina preventiva e etc",
    layout = widgets.Layout(width = "500px", height = "50px")
)

## Criando o botão de geração

In [7]:
generate_button = widgets.Button(
    description = "Gerar conteúdo"
)

## Exibindo resultado

In [8]:
output = widgets.Output()

### Definindo ação do botão

In [13]:
def generate_result(b):
  with output:
    output.clear_output()
    print("OOK!")

In [14]:
generate_button.on_click(generate_result)

## Exibindo os campos juntos na interface

In [50]:
def create_form():
  return widgets.VBox([
      topic,
      platform,
      tone,
      length,
      audience,
      cta,
      emoji,
      hashtags,
      keywords,
      generate_button,
      output
  ])

form = create_form()

## Conectando com a LLM

In [12]:
os.environ["GROQ_API_KEY"] = getpass.getpass()

··········


## Escolhendo o modelo

In [16]:
id_model = "llama3-8b-8192"

llm = ChatGroq(
    model = id_model,
    temperature = 0.7,
    max_tokens = None,
    timeout = None,
    max_retries = 2,
)

# Melhorando a exibição do resultado

In [ ]:
def show_res(res):
  from IPython.display import Markdown
  display(Markdown(res))

## Conclusão da aplicação

In [44]:
def llm_generate(llm, prompt):
  template = ChatPromptTemplate.from_messages([
      ("system", "Você é um redator profissional."),
      ("human", "{prompt}"),
  ])

  chain = template | llm | StrOutputParser()

  res = chain.invoke({"prompt": prompt})
  show_res(res)

In [56]:
def generate_result(b):
  with output:
    output.clear_output()

    prompt = f"""
    Escreva um texto com SEO otimizado sobre o tema "{topic.value}".
    Retorne em sua resposta apenas o texto final.
    - Onde será publicado: {platform.value}.
    - Tom: {tone.value}.
    - Publico alvo: {audience.value}.
    - Comprimento: {length.value}.
    - {"Inclua uma chamada para ação clara." if cta.value else "Não inclua chamada para ação"}
    - {"Inclua bastante emojis no texto" if emoji.value else "Não inlcua emojis no texto"}
    - {"Retorno ao final do texto hashtags relevantes." if hashtags.value else "Não inclua hashtags."}
    {"-Palavras-chave que devem estar presentes nesse texto (para SEO): " + keywords.value if keywords.value else ""}

    """

    try:
      res = llm_generate(llm, prompt)
      show_res(res)
    except Exception as e:
      print(f"Erro: {e}")

In [57]:
output = widgets.Output()
generate_button = widgets.Button(description = "Gerar conteúdo")
generate_button.on_click(generate_result)
form = create_form()

In [58]:
display(form)

# Construindo Streamlit

## Instalação do Streamlit

In [ ]:
!pip install -q streamlit
!npm install -q localtunnel
!pip install -q python-dotenv

## Instalação do ngrok


In [ ]:
!pip install pyngrok

In [83]:
from pyngrok import ngrok

In [ ]:
%%writefile .env
GROQ_API_KEY = SUA CHAVE AQUI

## Criação do arquivo da aplicação

In [87]:
%%writefile app.py
import streamlit as st
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

id_model = "llama3-8b-8192"

llm = ChatGroq(
    model = id_model,
    temperature = 0.7,
    max_tokens = None,
    timeout = None,
    max_retries = 2,
)

def llm_generate(llm, prompt):
  template = ChatPromptTemplate.from_messages([
      ("system", "Você é um redator profissional."),
      ("human", "{prompt}"),
  ])

  chain = template | llm | StrOutputParser()

  res = chain.invoke({"prompt": prompt})
  return res

st.set_page_config(page_title ="Gerador de conteúdo 🤖", page_icon = "🤖")
st.title("Gerador de conteúdo")

#Campos do formulario
topic = st.text_input("Tema:", placeholder = "Ex: Saúde mental, alimentação saudável, prevenção, etc.")
platform = st.selectbox("Plataforma:", ["Instagram", "Facebook", "Linkedln", "Blog", "E-mail"])
tone = st.selectbox("Tom:", ["Normal", "Informativo", "Inspirador", "Urgente", "Informal"])
length = st.selectbox("Tamanho:", ["Curto", "Médio", "Longo"])
audience = st.selectbox("Público-alvo:", ["Geral", "Jovens adultos", "Famílias", "Idosos", "Adolescentes"])
cta = st.checkbox("Incluir CTA")
emoji = st.checkbox("Incluir Emojis")
hashtags = st.checkbox("Incluir Hashtags")
keywords = st.text_area("Palavras-chave (SEO):", placeholder = "Ex: bem-estarm, medicina preventiva...")

if st.button("Gerar conteúdo"):
    prompt = f"""
    Escreva um texto com SEO otimizado sobre o tema "{topic}".
    Retorne em sua resposta apenas o texto final.
    - Onde será publicado: {platform}.
    - Tom: {tone}.
    - Publico alvo: {audience}.
    - Comprimento: {length}.
    - {"Inclua uma chamada para ação clara." if cta else "Não inclua chamada para ação"}
    - {"Inclua bastante emojis no texto" if emoji else "Não inlcua emojis no texto"}
    - {"Retorno ao final do texto hashtags relevantes." if hashtags else "Não inclua hashtags."}
    {"-Palavras-chave que devem estar presentes nesse texto (para SEO): " + keywords if keywords else ""}

    """

    try:
      res = llm_generate(llm, prompt)
      st.markdown(res)
    except Exception as e:
      print(f"Erro: {e}")

Overwriting app.py


## Execução do Streamlit

In [79]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!wget -q -O - ipv4.icanhazip.com

In [ ]:
!npx localtunnel --port 8501

### Execução com ngrok

In [ ]:
!ngrok config add-authtoken SUA CHAVE NGROK
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

public_url = ngrok.connect(8501)
public_url